In [ ]:
import os, json
import numpy as np

# -------------------------
# Load helpers
# -------------------------
def load_all_examples_with_prefix(files_with_prefix):
    """
    files_with_prefix: list of (prefix, jsonl_path)
    Returns {new_id: obj} where new_id = f"{prefix}_{old_id}"
    """
    all_examples = {}
    for prefix, path in files_with_prefix:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                old_id = obj.get("id")
                new_id = f"{prefix}_{old_id}"
                obj["id"] = new_id
                all_examples[new_id] = obj
    return all_examples


def _find_latest_run(sd_path, prefix):
    """
    sd_path: .../<prompt_subdir>
    prefix: "ua_run_" or "ru_run_"
    """
    runs = [
        d for d in os.listdir(sd_path)
        if d.startswith(prefix) and os.path.isdir(os.path.join(sd_path, d))
    ]
    if not runs:
        return None
    runs.sort()
    latest = runs[-1]
    fname = "ua_results.jsonl" if prefix.startswith("ua") else "ru_results.jsonl"
    fpath = os.path.join(sd_path, latest, fname)
    return fpath if os.path.isfile(fpath) else None


def collect_latest_runs_all_subdirs(model_dir):
    """
    model_dir: .../BASE_JUDGE/<model>
    Scans ALL first-level subdirectories (prompts), and picks latest ua_run_*/ru_run_*
    Returns:
      ua_list: [(subdir, ua_results.jsonl), ...]
      ru_list: [(subdir, ru_results.jsonl), ...]
    """
    ua_files, ru_files = [], []

    if not os.path.isdir(model_dir):
        raise FileNotFoundError(f"model_dir not found: {model_dir}")

    subdirs = sorted([
        d for d in os.listdir(model_dir)
        if os.path.isdir(os.path.join(model_dir, d))
    ])

    for sd in subdirs:
        sd_path = os.path.join(model_dir, sd)

        ua_path = _find_latest_run(sd_path, "ua_run_")
        ru_path = _find_latest_run(sd_path, "ru_run_")

        # keep only if both exist
        if ua_path and ru_path:
            ua_files.append((sd, ua_path))
            ru_files.append((sd, ru_path))

    return ua_files, ru_files


# -------------------------
# Binary direction labels (NO TIE)
# -------------------------
def scores_from_examples(examples: dict, score_key: str) -> dict:
    out = {}
    for ex_id, obj in examples.items():
        if score_key in obj and obj[score_key] is not None:
            out[ex_id] = float(obj[score_key])
    return out


def direction_labels_binary_from_examples(
    ua_examples: dict,
    ru_examples: dict,
    ua_key="score_ua_mt_vs_ua_human",
    ru_key="score_ru_mt_vs_ru_human",
    tie_break="RU",  # only if exactly equal
):
    """
    Returns {id: "UA"|"RU"} (no TIE).
    UA if ua_score < ru_score else RU (or tie_break on equality).
    """
    ua_scores = scores_from_examples(ua_examples, ua_key)
    ru_scores = scores_from_examples(ru_examples, ru_key)

    common = sorted(set(ua_scores) & set(ru_scores))
    labels = {}

    for i in common:
        ua = float(ua_scores[i])
        ru = float(ru_scores[i])
        if ua < ru:
            labels[i] = "UA"
        elif ru < ua:
            labels[i] = "RU"
        else:
            labels[i] = tie_break

    return labels


# -------------------------
# Compare two judges over ALL prompts (all subdirs) for a model
# -------------------------
def compare_judges_over_all_prompts(
    gpt_base_dir,
    llama_base_dir,
    model,
    ua_key="score_ua_mt_vs_ua_human",
    ru_key="score_ru_mt_vs_ru_human",
):
    """
    Goes over ALL prompt subdirs under each judge directory for a given model,
    loads latest ua/ru runs, makes binary UA/RU label per item,
    then compares labels on matching item IDs across judges.

    Returns a dict with confusion counts + agreement.
    """
    gpt_model_dir = os.path.join(gpt_base_dir, model)
    llama_model_dir = os.path.join(llama_base_dir, model)

    # GPT judge: load all latest files across all subdirs
    gpt_ua_files, gpt_ru_files = collect_latest_runs_all_subdirs(gpt_model_dir)
    gpt_ua = load_all_examples_with_prefix(gpt_ua_files)
    gpt_ru = load_all_examples_with_prefix(gpt_ru_files)
    labels_gpt = direction_labels_binary_from_examples(gpt_ua, gpt_ru, ua_key=ua_key, ru_key=ru_key)

    # Llama judge: load all latest files across all subdirs
    ll_ua_files, ll_ru_files = collect_latest_runs_all_subdirs(llama_model_dir)
    ll_ua = load_all_examples_with_prefix(ll_ua_files)
    ll_ru = load_all_examples_with_prefix(ll_ru_files)
    labels_llama = direction_labels_binary_from_examples(ll_ua, ll_ru, ua_key=ua_key, ru_key=ru_key)

    # Match items across judges
    common = sorted(set(labels_gpt) & set(labels_llama))
    if not common:
        raise ValueError("No overlapping items across judges (after prefixing).")

    # Confusion counts
    # rows = GPT, cols = Llama
    conf = {
        ("RU","RU"): 0,
        ("RU","UA"): 0,
        ("UA","RU"): 0,
        ("UA","UA"): 0,
    }
    for i in common:
        conf[(labels_gpt[i], labels_llama[i])] += 1

    n = len(common)
    both_ru = conf[("RU","RU")]
    both_ua = conf[("UA","UA")]
    disagree = conf[("RU","UA")] + conf[("UA","RU")]
    pct_agree = (both_ru + both_ua) / n

    ex_both_ua = [i for i in common if labels_gpt[i] == "UA" and labels_llama[i] == "UA"]
    ex_both_ru = [i for i in common if labels_gpt[i] == "RU" and labels_llama[i] == "RU"]
    ex_gpt_ua_ll_ru = [i for i in common if labels_gpt[i] == "UA" and labels_llama[i] == "RU"]
    ex_gpt_ru_ll_ua = [i for i in common if labels_gpt[i] == "RU" and labels_llama[i] == "UA"]

    return {
        "model": model,
        "n_common": n,
        "pct_agreement": float(pct_agree),
        "counts": {
            "both_UA": int(both_ua),
            "both_RU": int(both_ru),
            "disagree": int(disagree),
            "gpt_UA_llama_RU": int(conf[("UA","RU")]),
            "gpt_RU_llama_UA": int(conf[("RU","UA")]),
        },
        "confusion_rows_gpt_cols_llama": {
            "RU": {"RU": int(conf[("RU","RU")]), "UA": int(conf[("RU","UA")])},
            "UA": {"RU": int(conf[("UA","RU")]), "UA": int(conf[("UA","UA")])},
        },
        "example_ids": {
            "both_UA": ex_both_ua,
            "both_RU": ex_both_ru,
            "gpt_UA_llama_RU": ex_gpt_ua_ll_ru,
            "gpt_RU_llama_UA": ex_gpt_ru_ll_ua,
        },
        "debug": {
            "gpt_num_prompt_subdirs_used": len(gpt_ua_files),   # only subdirs that had BOTH UA+RU latest files
            "llama_num_prompt_subdirs_used": len(ll_ua_files),
            "gpt_prompt_subdirs_used": [p for p, _ in gpt_ua_files],
            "llama_prompt_subdirs_used": [p for p, _ in ll_ua_files],
        }
    }


In [ ]:
import os, json
import numpy as np

# -------------------------
# Load helpers
# -------------------------
def load_all_examples_with_prefix(files_with_prefix):
    """
    files_with_prefix: list of (prefix, jsonl_path)
    Returns {new_id: obj} where new_id = f"{prefix}_{old_id}"
    """
    all_examples = {}
    for prefix, path in files_with_prefix:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                old_id = obj.get("id")
                new_id = f"{prefix}_{old_id}"
                obj["id"] = new_id
                all_examples[new_id] = obj
    return all_examples


def _find_latest_run(sd_path, prefix):
    runs = [
        d for d in os.listdir(sd_path)
        if d.startswith(prefix) and os.path.isdir(os.path.join(sd_path, d))
    ]
    if not runs:
        return None
    runs.sort()
    latest = runs[-1]
    fname = "ua_results.jsonl" if prefix.startswith("ua") else "ru_results.jsonl"
    fpath = os.path.join(sd_path, latest, fname)
    return fpath if os.path.isfile(fpath) else None


def collect_latest_runs_all_subdirs(model_dir):
    """
    model_dir: .../BASE_JUDGE/<model>
    Scans ALL first-level subdirectories (prompts), and picks latest ua_run_*/ru_run_*
    Returns:
      ua_list: [(subdir, ua_results.jsonl), ...]
      ru_list: [(subdir, ru_results.jsonl), ...]
    Keeps only subdirs where BOTH UA and RU latest files exist.
    """
    ua_files, ru_files = [], []

    if not os.path.isdir(model_dir):
        raise FileNotFoundError(f"model_dir not found: {model_dir}")

    subdirs = sorted([
        d for d in os.listdir(model_dir)
        if os.path.isdir(os.path.join(model_dir, d))
    ])

    for sd in subdirs:
        sd_path = os.path.join(model_dir, sd)
        ua_path = _find_latest_run(sd_path, "ua_run_")
        ru_path = _find_latest_run(sd_path, "ru_run_")

        if ua_path and ru_path:
            ua_files.append((sd, ua_path))
            ru_files.append((sd, ru_path))

    return ua_files, ru_files


# -------------------------
# Direction labels (binary RU/UA) but IGNORE exact ties (diff == 0.0)
# -------------------------
def scores_from_examples(examples: dict, score_key: str) -> dict:
    out = {}
    for ex_id, obj in examples.items():
        if score_key in obj and obj[score_key] is not None:
            out[ex_id] = float(obj[score_key])
    return out


def direction_labels_binary_ignore_zero_diff(
    ua_examples: dict,
    ru_examples: dict,
    ua_key="score_ua_mt_vs_ua_human",
    ru_key="score_ru_mt_vs_ru_human",
    zero_tol=0.0,
):
    """
    Returns {id: "UA"|"RU"} with NO TIE label.
    Ignores items where (ua_score - ru_score) == 0.0 (within zero_tol).

    UA if ua_score < ru_score
    RU if ru_score < ua_score
    skip if abs(ua_score - ru_score) <= zero_tol
    """
    ua_scores = scores_from_examples(ua_examples, ua_key)
    ru_scores = scores_from_examples(ru_examples, ru_key)

    common = sorted(set(ua_scores) & set(ru_scores))
    labels = {}

    for i in common:
        ua = float(ua_scores[i])
        ru = float(ru_scores[i])
        d = ua - ru
        if abs(d) <= zero_tol:
            continue  # ignore exact tie / no preference
        labels[i] = "UA" if d < 0 else "RU"

    return labels


# -------------------------
# Compare two judges over ALL prompts for a model (ignore zero-diff items)
# -------------------------
def compare_judges_over_all_prompts_ignore_zero(
    gpt_base_dir,
    llama_base_dir,
    model,
    ua_key="score_ua_mt_vs_ua_human",
    ru_key="score_ru_mt_vs_ru_human",
    zero_tol=0.0,
):
    gpt_model_dir = os.path.join(gpt_base_dir, model)
    llama_model_dir = os.path.join(llama_base_dir, model)

    # GPT judge: load all latest files across all prompt subdirs
    gpt_ua_files, gpt_ru_files = collect_latest_runs_all_subdirs(gpt_model_dir)
    gpt_ua = load_all_examples_with_prefix(gpt_ua_files)
    gpt_ru = load_all_examples_with_prefix(gpt_ru_files)
    labels_gpt = direction_labels_binary_ignore_zero_diff(
        gpt_ua, gpt_ru, ua_key=ua_key, ru_key=ru_key, zero_tol=zero_tol
    )

    # Llama judge
    ll_ua_files, ll_ru_files = collect_latest_runs_all_subdirs(llama_model_dir)
    ll_ua = load_all_examples_with_prefix(ll_ua_files)
    ll_ru = load_all_examples_with_prefix(ll_ru_files)
    labels_llama = direction_labels_binary_ignore_zero_diff(
        ll_ua, ll_ru, ua_key=ua_key, ru_key=ru_key, zero_tol=zero_tol
    )

    # Match items across judges AFTER filtering zero-diff inside each judge
    common = sorted(set(labels_gpt) & set(labels_llama))
    if not common:
        raise ValueError("No overlapping non-zero-preference items across judges (after filtering).")

    # Confusion counts (rows=GPT, cols=Llama)
    conf = {
        ("RU", "RU"): 0,
        ("RU", "UA"): 0,
        ("UA", "RU"): 0,
        ("UA", "UA"): 0,
    }
    for i in common:
        conf[(labels_gpt[i], labels_llama[i])] += 1

    n = len(common)
    both_ru = conf[("RU", "RU")]
    both_ua = conf[("UA", "UA")]
    disagree = conf[("RU", "UA")] + conf[("UA", "RU")]
    pct_agree = (both_ru + both_ua) / n

    # Example IDs (optional)
    ex_both_ua = [i for i in common if labels_gpt[i] == "UA" and labels_llama[i] == "UA"]
    ex_both_ru = [i for i in common if labels_gpt[i] == "RU" and labels_llama[i] == "RU"]
    ex_gpt_ua_ll_ru = [i for i in common if labels_gpt[i] == "UA" and labels_llama[i] == "RU"]
    ex_gpt_ru_ll_ua = [i for i in common if labels_gpt[i] == "RU" and labels_llama[i] == "UA"]

    return {
        "model": model,
        "zero_tol": float(zero_tol),
        "n_common_nonzero": n,
        "pct_agreement": float(pct_agree),
        "counts": {
            "both_UA": int(both_ua),
            "both_RU": int(both_ru),
            "disagree": int(disagree),
            "gpt_UA_llama_RU": int(conf[("UA", "RU")]),
            "gpt_RU_llama_UA": int(conf[("RU", "UA")]),
        },
        "confusion_rows_gpt_cols_llama": {
            "RU": {"RU": int(conf[("RU", "RU")]), "UA": int(conf[("RU", "UA")])},
            "UA": {"RU": int(conf[("UA", "RU")]), "UA": int(conf[("UA", "UA")])},
        },
        "example_ids": {
            "both_UA": ex_both_ua,
            "both_RU": ex_both_ru,
            "gpt_UA_llama_RU": ex_gpt_ua_ll_ru,
            "gpt_RU_llama_UA": ex_gpt_ru_ll_ua,
        },
        "debug": {
            "gpt_num_prompt_subdirs_used": len(gpt_ua_files),
            "llama_num_prompt_subdirs_used": len(ll_ua_files),
            "gpt_prompt_subdirs_used": [p for p, _ in gpt_ua_files],
            "llama_prompt_subdirs_used": [p for p, _ in ll_ua_files],
            "gpt_labels_nonzero": len(labels_gpt),
            "llama_labels_nonzero": len(labels_llama),
        }
    }



In [ ]:
import os, json
import numpy as np

# ============================================================
# 1) Latest-run discovery + reading *_summary.json (per prompt)
# ============================================================

def _find_latest_run_dir(prompt_dir: str, prefix: str) -> str | None:
    if not os.path.isdir(prompt_dir):
        return None
    runs = [
        d for d in os.listdir(prompt_dir)
        if d.startswith(prefix) and os.path.isdir(os.path.join(prompt_dir, d))
    ]
    if not runs:
        return None
    runs.sort()
    return os.path.join(prompt_dir, runs[-1])


def _read_json(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _extract_mean(summary: dict, side: str):
    """
    side: "ru" or "ua"
    Accepts either *_POV or *_GT style keys.
    Returns (mean, n, key_used) or (None, None, None).
    """
    if not isinstance(summary, dict):
        return None, None, None

    n = summary.get("n", None)
    try:
        n = int(n) if n is not None else None
    except Exception:
        n = None

    keys = list(summary.keys())
    keys_l = [k.lower() for k in keys]

    if side == "ru":
        prefer = ["mean_score_ru_mt_vs_ru_pov", "mean_score_ru_mt_vs_ru_gt"]
        prefix = "mean_score_ru"
        contains = "mt_vs_ru"
    else:
        prefer = ["mean_score_ua_mt_vs_ua_pov", "mean_score_ua_mt_vs_ua_gt"]
        prefix = "mean_score_ua"
        contains = "mt_vs_ua"

    # 1) preferred keys
    for want in prefer:
        if want in keys_l:
            k = keys[keys_l.index(want)]
            return float(summary[k]), n, k

    # 2) fallback: any mean_score_{side} containing mt_vs_{side}
    candidates = []
    for k in keys:
        kl = k.lower()
        if kl.startswith(prefix) and contains in kl:
            candidates.append(k)

    if candidates:
        # prefer pov > gt if present
        pov = [k for k in candidates if "pov" in k.lower()]
        gt  = [k for k in candidates if "gt"  in k.lower()]
        pick = pov[0] if pov else (gt[0] if gt else candidates[0])
        try:
            return float(summary[pick]), n, pick
        except Exception:
            return None, n, pick

    return None, n, None


def load_latest_prompt_summary_pair(judge_base_dir: str, model: str, prompt_subdir: str):
    """
    Reads:
      .../<judge_base>/<model>/<prompt_subdir>/ru_run_*/ru_summary.json
      .../<judge_base>/<model>/<prompt_subdir>/ua_run_*/ua_summary.json

    Returns:
      dict with means + preference, or None if missing
    """
    prompt_dir = os.path.join(judge_base_dir, model, prompt_subdir)
    ru_run = _find_latest_run_dir(prompt_dir, "ru_run_")
    ua_run = _find_latest_run_dir(prompt_dir, "ua_run_")
    if not ru_run or not ua_run:
        return None

    ru_sum_path = os.path.join(ru_run, "ru_summary.json")
    ua_sum_path = os.path.join(ua_run, "ua_summary.json")
    if not (os.path.isfile(ru_sum_path) and os.path.isfile(ua_sum_path)):
        return None

    ru_sum = _read_json(ru_sum_path)
    ua_sum = _read_json(ua_sum_path)

    mean_ru, n_ru, key_ru = _extract_mean(ru_sum, side="ru")
    mean_ua, n_ua, key_ua = _extract_mean(ua_sum, side="ua")

    if mean_ru is None or mean_ua is None:
        return None

    # preference per PROMPT SUBDIR: smaller mean distance is "closer" to that side
    if mean_ua < mean_ru:
        pref = "UA"
    elif mean_ru < mean_ua:
        pref = "RU"
    else:
        pref = "OK"  # exact equality at mean level (rare)

    return {
        "prompt": prompt_subdir,
        "mean_ru": float(mean_ru),
        "mean_ua": float(mean_ua),
        "pref": pref,
        "n_ru": n_ru,
        "n_ua": n_ua,
        "key_ru": key_ru,
        "key_ua": key_ua,
        "ru_summary": ru_sum_path,
        "ua_summary": ua_sum_path,
    }


# ============================================================
# 2) Build per-prompt preference labels for a judge
# ============================================================

def list_prompt_subdirs(judge_base_dir: str, model: str):
    """
    Returns all first-level prompt subdirs under .../<judge_base>/<model>
    (e.g., neutral_1, twitter_1, news_article_2, etc.)
    """
    root = os.path.join(judge_base_dir, model)
    if not os.path.isdir(root):
        raise FileNotFoundError(f"Model root not found: {root}")

    return sorted([
        d for d in os.listdir(root)
        if os.path.isdir(os.path.join(root, d))
    ])


def prompt_labels_from_summaries(judge_base_dir: str, model: str, ignore_ok=True):
    """
    Returns:
      labels:  {prompt_subdir: "RU"|"UA"}   (or includes "OK" if ignore_ok=False)
      details: {prompt_subdir: {...}}       (means, paths, keys used)
    """
    labels = {}
    details = {}

    for prompt in list_prompt_subdirs(judge_base_dir, model):
        rec = load_latest_prompt_summary_pair(judge_base_dir, model, prompt)
        if rec is None:
            continue
        details[prompt] = rec
        if ignore_ok and rec["pref"] == "OK":
            continue
        labels[prompt] = rec["pref"]

    return labels, details


# ============================================================
# 3) Cohen's kappa over per-prompt labels (RU/UA only)
# ============================================================

def cohen_kappa_binary_from_prompt_labels(labels_a: dict, labels_b: dict):
    """
    labels_a/b: {prompt_subdir: "RU"|"UA"}
    Computes kappa over common prompts.
    """
    common = sorted(set(labels_a) & set(labels_b))
    if not common:
        raise ValueError("No overlapping prompt labels between judges.")

    cats = ["RU", "UA"]
    idx = {c: i for i, c in enumerate(cats)}

    M = np.zeros((2, 2), dtype=float)  # rows=A, cols=B
    for p in common:
        a = labels_a[p]
        b = labels_b[p]
        if a not in idx or b not in idx:
            continue
        M[idx[a], idx[b]] += 1.0

    n = M.sum()
    if n == 0:
        raise ValueError("No usable RU/UA labels in common prompts (after filtering).")

    po = float(np.trace(M) / n)
    pa = M.sum(axis=1) / n
    pb = M.sum(axis=0) / n
    pe = float(np.sum(pa * pb))
    kappa = (po - pe) / (1.0 - pe) if (1.0 - pe) > 1e-12 else 0.0

    return {
        "n_common_prompts": int(n),
        "prompts": common,
        "pct_agreement": po,
        "kappa": float(kappa),
        "confusion_rows_A_cols_B": {
            "RU": {"RU": int(M[idx["RU"], idx["RU"]]), "UA": int(M[idx["RU"], idx["UA"]])},
            "UA": {"RU": int(M[idx["UA"], idx["RU"]]), "UA": int(M[idx["UA"], idx["UA"]])},
        }
    }

